# 04 - Entrainement du Modele ML

Ce notebook montre comment entrainer un modele de prediction de risque.

## Pipeline ML
1. Extraction des features depuis Neo4j
2. Preparation des donnees
3. Entrainement RandomForest
4. Evaluation et sauvegarde

In [ ]:
import os
from neo4j import GraphDatabase
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import joblib

In [ ]:
NEO4J_URI = os.environ.get('NEO4J_URI', 'bolt://neo4j:7687')
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(os.environ.get('NEO4J_USER', 'neo4j'),
          os.environ['NEO4J_PASSWORD'])
)

def query(cypher):
    with driver.session() as session:
        result = session.run(cypher)
        return pd.DataFrame([dict(r) for r in result])

## 1. Extraction des features

In [ ]:
df = query("""
MATCH (p:Patient)
OPTIONAL MATCH (p)-[:LIVES_IN]->(l:Location)
OPTIONAL MATCH (p)-[:INFECTED_BY]-(contact)
OPTIONAL MATCH (p)-[:HAS_DISEASE]->(d:Disease)
WITH p, l, count(contact) AS contacts, 
     CASE WHEN d IS NOT NULL THEN 1 ELSE 0 END AS infected
RETURN p.patient_id AS id,
       p.age AS age,
       p.gender AS gender,
       p.comorbidities AS comorbidities,
       p.vaccination_status AS vaccination,
       coalesce(l.population_density, 0) AS density,
       contacts,
       infected AS label
LIMIT 5000
""")
print(f"Echantillon: {len(df)} patients")
df.head()

## 2. Preparation des donnees

In [ ]:
# Encodage des variables categoriques
encoders = {}
for col in ['gender', 'comorbidities', 'vaccination']:
    encoders[col] = LabelEncoder()
    df[col + '_enc'] = encoders[col].fit_transform(df[col].fillna('unknown'))

# Features et label
feature_cols = ['age', 'gender_enc', 'comorbidities_enc', 'vaccination_enc', 'density', 'contacts']
X = df[feature_cols].fillna(0)
y = df['label']

print(f"Shape X: {X.shape}, y: {y.shape}")
print(f"Distribution label: {y.value_counts().to_dict()}")

In [ ]:
# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

## 3. Entrainement

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
print("Modele entraine")

## 4. Evaluation

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Non infecte', 'Infecte']))

In [ ]:
# Importance des features
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
importance

In [ ]:
# Sauvegarde
joblib.dump({'model': model, 'encoders': encoders}, 'risk_model.pkl')
print("Modele sauvegarde")
driver.close()